<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-03-arvores/Aula%2028%20-%20Apresenta%C3%A7%C3%A3o%20e%20Simula%C3%A7%C3%A3o%20do%20M%C3%B3dulo%203%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 28 - Notebook: Simulação Integrada do Módulo 3 (`SCADACoreModulo3`)

Este notebook integra os algoritmos de **Árvores (Aulas 19 a 27)** em um simulador único da Estação de Reabastecimento de Hidrogênio (`SCADA-Core`), validando:
- **Cenário A:** Abastecimento nominal completo SAE J2601 (`50 bar -> 700 bar`);
- **Cenário B:** Corte de fibra (`Failover` RSTP) + vazamento de $\text{H}_2$ (`AT-301 > 25% LIE`) com preempção reativa e diagnóstico automático.

In [ ]:
import heapq
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from enum import Enum

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print("Módulos base carregados para o SCADACoreModulo3.")

### 1. Subsistemas Integrados: RSTP (Aula 23), Huffman (Aula 20), BDT (Aula 21) e Diagnóstico (Aula 27)

In [ ]:
def verificar_rede_rstp(corte_backbone: bool = False) -> bool:
    G = nx.Graph()
    enlaces = [
        ("SW-SCADA", "SW-S100", 2000),
        ("SW-SCADA", "SW-SIS",  20000),
        ("SW-S100",  "SW-S200", 20000),
        ("SW-S200",  "SW-S300", 20000),
        ("SW-S300",  "SW-SIS",  20000),
    ]
    for u, v, w in enlaces:
        G.add_edge(u, v, weight=w)
    if corte_backbone:
        G.remove_edge("SW-SCADA", "SW-S100")
    T = nx.minimum_spanning_tree(G)
    return nx.is_connected(T) and T.number_of_edges() == G.number_of_nodes() - 1

# Tabela de Huffman (Aula 20) para o vetor [PT-101, TT-101, AT-101/301, PT-201, ESD-100]
HUFFMAN_CODES = {
    "00000": "0",      # Nominal
    "00010": "10",     # Transiente PT-201
    "10000": "110",    # Sobrepressao PT-101
    "01000": "1110",   # Sobretemperatura TT-101
    "00100": "11110",  # Vazamento H2 > 25% LIE
    "00001": "11111",  # ESD-100
}

def codificar_telemetria_huffman(pt101, tt101, at_max, pt201, esd100):
    bits = f"{int(pt101>400)}{int(tt101>85)}{int(at_max>25)}{int(pt201>=850)}{int(esd100==1)}"
    return HUFFMAN_CODES.get(bits, "11111"), bits

def avaliar_bdt_sis(pt101, tt101, at_max, esd100) -> bool:
    """Retorna True se seguro, False se houver Trip SIS (O(1))."""
    if esd100 == 1 or pt101 > 400.0 or tt101 > 85.0 or at_max > 25.0:
        return False
    return True

def diagnosticar_cart(pt101, tt101, at_max) -> str:
    """Regras induzidas da Arvore de Decisao CART (Aula 27)."""
    if at_max > 15.0:
        return "Vazamento_H2"
    if pt101 > 380.0:
        return "Sobrepressao_Cascata"
    if tt101 > 55.0:
        return "Falha_Chiller_Termica"
    return "Nominal"

### 2. Classe `SCADACoreModulo3` e Execução dos Cenários de Estresse

In [ ]:
class SCADACoreModulo3:
    def simular_ciclo(self, nome: str, corte_fibra: bool = False, tick_vazamento: int = None):
        rede_ok = verificar_rede_rstp(corte_backbone=corte_fibra)
        pt_301, tt_201, pt_101, tt_101, at_301, esd_100 = 50.0, -40.0, 340.0, 26.0, 2.0, 0
        bits_fixos, bits_huff = 0, 0
        logs = []

        for tick in range(1, 16):
            if tick_vazamento is not None and tick >= tick_vazamento:
                at_301 = 34.0  # Disparo > 25% LIE

            cod, raw = codificar_telemetria_huffman(pt_101, tt_101, at_301, 720.0, esd_100)
            bits_fixos += 5
            bits_huff += len(cod)

            seguro = avaliar_bdt_sis(pt_101, tt_101, at_301, esd_100)
            diag = diagnosticar_cart(pt_101, tt_101, at_301)

            if not seguro:
                xv_301, alarme, status = 0, 1, "TRIP_SIS"
            else:
                xv_301, alarme, status = 1, 0, ("CONCLUIDO" if pt_301 >= 700.0 else "ABASTECENDO")
                if pt_301 < 700.0:
                    pt_301 = min(700.0, pt_301 + 55.0)

            banco = "TK-101 (LP)" if pt_301 < 350 else ("TK-102 (MP)" if pt_301 < 650 else "TK-103 (HP)")
            logs.append({
                "Tick": tick, "Rede_RSTP": rede_ok, "PT_301_bar": pt_301,
                "AT_301_LIE": at_301, "Banco_XV104": banco, "XV_301": xv_301,
                "Status": status, "Diagnostico_CART": diag
            })
            if status in ("TRIP_SIS", "CONCLUIDO"):
                break

        economia = (1.0 - bits_huff / bits_fixos) * 100.0
        print(f"=== {nome} | Rede RSTP OK: {rede_ok} | Economia Huffman: {economia:.1f}% ===")
        return pd.DataFrame(logs)

scada = SCADACoreModulo3()
df_a = scada.simular_ciclo("Cenário A (Nominal 700 bar)")
df_b = scada.simular_ciclo("Cenário B (Corte Fibra + Vazamento AT-301)", corte_fibra=True, tick_vazamento=8)

display(df_a.tail(4))
display(df_b.tail(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df_a["Tick"], df_a["PT_301_bar"], marker="o", color="green", label="Cenário A: Nominal (700 bar)")
ax.plot(df_b["Tick"], df_b["PT_301_bar"], marker="x", color="red", linestyle="--", label="Cenário B: Preempção Vazamento AT-301 > 25% LIE")
ax.axhline(700.0, color="purple", linestyle=":", label="Setpoint FCEV (700 bar)")
ax.set_title("Simulação Integrada Módulo 3 — Pressão no Dispensador DISP-300")
ax.set_xlabel("Tick de Varredura SCADA")
ax.set_ylabel("Pressão PT-301 [bar]")
ax.legend()
plt.tight_layout()
plt.show()